# Phase 1b Data Upgrade: A Larger OSHA Accident Dataset

In my first pass I worked from a small snapshot of the OSHA enforcement archive that held only 774 accident records. That left me with about 266 construction cases, which is too few for baselines I can trust.

The Department of Labor publishes the full enforcement data, and it holds far more accident investigations, each linked to an inspection record that carries an industry code. In this notebook I load that larger source and rebuild my construction accident table from it.

The steps are:

1. Check the database file and the tables I need
2. Load the accident and injury records
3. Link each accident to its inspection to get industry codes
4. Identify construction accidents using three independent rules
5. Attach the narrative text
6. Audit the columns for leakage and save a clean table

The source is the OSHA enforcement data from the Department of Labor, accessed through a public mirror of the DOL Enforcement Data Catalog.

In [1]:
from pathlib import Path
import re
import sqlite3
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 100)

RANDOM_STATE = 42

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
DB_PATH = RAW / "osha_enforcement.db"

print("Project root:", ROOT)
print("Database found:", DB_PATH.exists())
if DB_PATH.exists():
    print("Database size (GB):", round(DB_PATH.stat().st_size / 1e9, 2))

Project root: c:\Users\sabin\constructiq
Database found: True
Database size (GB): 6.49


## 1. Checking the database

The data comes as a single SQLite database file. I connect to it, list the tables and count the rows in the ones I need.

In [2]:
conn = sqlite3.connect(str(DB_PATH))

tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)["name"].tolist()
print("Tables in the database:", len(tables))

for t in ["accident", "accident_injury", "accident_abstract", "inspection", "violation"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t:20s} {n:>12,} rows")

print()
print("Inspection columns:")
print(pd.read_sql("PRAGMA table_info(inspection)", conn)["name"].tolist())

Tables in the database: 25
accident                  165,801 rows
accident_injury           231,809 rows
accident_abstract       1,151,160 rows
inspection              5,184,668 rows
violation              13,237,726 rows

Inspection columns:
['activity_nr', 'reporting_id', 'state_flag', 'estab_name', 'site_address', 'site_city', 'site_state', 'site_zip', 'owner_type', 'owner_code', 'adv_notice', 'safety_hlth', 'sic_code', 'naics_code', 'insp_type', 'insp_scope', 'why_no_insp', 'union_status', 'safety_manuf', 'safety_const', 'safety_marit', 'health_manuf', 'health_const', 'health_marit', 'migrant', 'mail_street', 'mail_city', 'mail_state', 'mail_zip', 'host_est_key', 'nr_in_estab', 'open_date', 'case_mod_date', 'close_conf_date', 'close_case_date', 'ld_dt']


## 2. Loading accidents and injuries

The accident table holds one row per accident. The injury table holds one row per injured person, and it carries the severity label (`degree_of_inj`) and the inspection number that links each injury to an inspection.

In [3]:
accident = pd.read_sql("SELECT * FROM accident", conn)
injury = pd.read_sql("SELECT * FROM accident_injury", conn)

for d in (accident, injury):
    d["summary_nr"] = pd.to_numeric(d["summary_nr"], errors="coerce").astype("Int64")

injury["rel_insp_nr"] = pd.to_numeric(injury["rel_insp_nr"], errors="coerce").astype("Int64")
injury["degree_of_inj"] = pd.to_numeric(injury["degree_of_inj"], errors="coerce")

print("Accidents:", accident.shape)
print("Injury rows:", injury.shape)
print()
print("Severity codes across all industries:")
print(injury["degree_of_inj"].value_counts(dropna=False).sort_index())

Accidents: (165801, 16)
Injury rows: (231809, 21)

Severity codes across all industries:
degree_of_inj
0.0     5436
1.0    93068
2.0    99885
3.0    33338
NaN       82
Name: count, dtype: int64


## 3. Linking each injury to its inspection

Each injury row has an inspection number. The inspection table is large, so I read it in chunks and keep only the inspections that are linked to an accident. It gives me the industry codes (NAICS and SIC) and the state.

In [4]:
wanted = set(injury["rel_insp_nr"].dropna().astype("int64"))
print("Distinct inspections linked to injuries:", len(wanted))

cols = "activity_nr, naics_code, sic_code, safety_const, site_state, open_date, insp_type"
parts = []
for chunk in pd.read_sql(f"SELECT {cols} FROM inspection", conn, chunksize=500_000):
    chunk["activity_nr"] = pd.to_numeric(chunk["activity_nr"], errors="coerce")
    parts.append(chunk[chunk["activity_nr"].isin(wanted)])

insp = pd.concat(parts, ignore_index=True).drop_duplicates("activity_nr")
insp["activity_nr"] = insp["activity_nr"].astype("int64")

print("Inspections found:", len(insp), "of", len(wanted))
print("Share with a NAICS code:", round(insp["naics_code"].notna().mean(), 3))
print("Share with a SIC code:", round(insp["sic_code"].notna().mean(), 3))
insp.head()

Distinct inspections linked to injuries: 194060
Inspections found: 193976 of 194060
Share with a NAICS code: 1.0
Share with a SIC code: 1.0


,activity_nr,naics_code,sic_code,safety_const,site_state,open_date,insp_type
0,102384,0,5171,,NY,1984-03-21 00:00:00+00:00,A
1,102814,0,1795,X,NY,1984-03-27 00:00:00+00:00,A
2,117192,0,2951,,NJ,1984-02-06 00:00:00+00:00,A
3,117796,0,4911,,NJ,1984-01-23 00:00:00+00:00,A
4,121715,0,4785,,NJ,1983-12-15 00:00:00+00:00,A


## 4. Finding construction accidents

There is no single construction flag, so I use independent rules and compare them:

1. **NAICS rule:** the inspected business has a NAICS code starting with 23, which is the construction sector
2. **SIC rule:** the older SIC code starts with 15, 16 or 17, the construction divisions used before NAICS
3. **Citation rule:** the inspection cited at least one standard from the 1926 series, OSHA's construction standards
4. **Keyword rule:** my earlier word search on the description, kept only for comparison because it was noisy

My final definition of construction is any of the first three rules. I do not use the keyword rule to select rows.

In [5]:
# Inspections that cited at least one construction standard 
cited = pd.read_sql("SELECT DISTINCT activity_nr FROM violation WHERE TRIM(standard) LIKE '1926%'", conn)
cited_set = set(pd.to_numeric(cited["activity_nr"], errors="coerce").dropna().astype("int64"))
print("Inspections that cited a 1926 standard:", len(cited_set))

Inspections that cited a 1926 standard: 1170763


In [6]:
acc_cols = ["summary_nr", "event_date", "event_desc", "event_keyword", "const_end_use",
            "build_stories", "nonbuild_ht", "project_cost", "project_type", "fatality"]

base = injury.merge(insp, left_on="rel_insp_nr", right_on="activity_nr", how="left")
base = base.merge(accident[acc_cols], on="summary_nr", how="left")

naics = base["naics_code"].astype("string").str.strip()
sic = base["sic_code"].astype("string").str.strip()
kw_text = (base["event_desc"].fillna("") + " " + base["event_keyword"].fillna("")).str.lower()

KEYWORDS = ["ladder", "scaffold", "scaffolding", "crane", "excavation", "roof",
            "construction", "concrete", "framing", "drywall", "rebar", "trench",
            "fall protection", "elevated work platform", "aerial lift"]

base["rule_naics"] = naics.str.startswith("23").fillna(False).astype(bool)
base["rule_sic"] = sic.str[:2].isin(["15", "16", "17"]).fillna(False).astype(bool)
base["rule_cited_1926"] = base["rel_insp_nr"].isin(cited_set).fillna(False).astype(bool)
base["rule_keyword"] = kw_text.str.contains("|".join(KEYWORDS), na=False)

base = base[base["degree_of_inj"].isin([1, 2, 3])].copy()
base["is_construction"] = base["rule_naics"] | base["rule_sic"] | base["rule_cited_1926"]

print("Labeled injury rows, all industries:", len(base))
print()
print("Rows flagged by each rule:")
print(base[["rule_naics", "rule_sic", "rule_cited_1926", "rule_keyword"]].sum())

Labeled injury rows, all industries: 226291

Rows flagged by each rule:
rule_naics         32604
rule_sic           55945
rule_cited_1926    36651
rule_keyword       62494
dtype: int64


In [7]:
label_map = {1: "Fatality", 2: "Hospitalized", 3: "Not hospitalized"}

print("NAICS rule versus citation rule:")
print(pd.crosstab(base["rule_naics"], base["rule_cited_1926"]))
print()

for name, mask in [
    ("NAICS rule", base["rule_naics"]),
    ("SIC rule", base["rule_sic"]),
    ("Citation rule", base["rule_cited_1926"]),
    ("My final definition (NAICS or SIC or citation)", base["is_construction"]),
    ("Keyword rule only, for comparison", base["rule_keyword"]),
]:
    sub = base[mask]
    print(name)
    print("  rows:", len(sub), "| unique accidents:", sub["summary_nr"].nunique())
    print("  ", sub["degree_of_inj"].map(label_map).value_counts().to_dict())
    print()

NAICS rule versus citation rule:
rule_cited_1926   False  True 
rule_naics                    
False            170479  23208
True              19161  13443

NAICS rule
  rows: 32604 | unique accidents: 29540
   {'Hospitalized': 15800, 'Fatality': 14283, 'Not hospitalized': 2521}

SIC rule
  rows: 55945 | unique accidents: 46564
   {'Fatality': 26889, 'Hospitalized': 23102, 'Not hospitalized': 5954}

Citation rule
  rows: 36651 | unique accidents: 30123
   {'Fatality': 21769, 'Hospitalized': 11916, 'Not hospitalized': 2966}

My final definition (NAICS or SIC or citation)
  rows: 74523 | unique accidents: 63180
   {'Fatality': 35882, 'Hospitalized': 31437, 'Not hospitalized': 7204}

Keyword rule only, for comparison
  rows: 62494 | unique accidents: 55263
   {'Hospitalized': 29651, 'Fatality': 27429, 'Not hospitalized': 5414}



In [8]:
print("Samples flagged by my final definition but NOT by the keyword rule:")
print(base[base["is_construction"] & ~base["rule_keyword"]]["event_desc"].dropna().sample(8, random_state=RANDOM_STATE).to_string(index=False))
print()
print("Samples flagged by the keyword rule only (not by my final definition):")
print(base[~base["is_construction"] & base["rule_keyword"]]["event_desc"].dropna().sample(8, random_state=RANDOM_STATE).to_string(index=False))

Samples flagged by my final definition but NOT by the keyword rule:
Employee is struck by motor vehicle, receives multiple fract
   2 EMPLOYEES KILLED, ONE INJURED WHEN PAINT FUMES EXPLODED
                 Employee injured while working in a culvert
                          TWO BUILDINGS COLLAPSED ON WORKERS
                      Employee's Finger Amputated in Machine
        Employee Suffers Burns When Lift Contacts Power Line
           TWO EMPLOYEES INJURED WHEN STRUCK BY FALLING ROCK
                         Employee Is Burned by Gasoline Fire

Samples flagged by the keyword rule only (not by my final definition):
          Employee injured from 14 foot fall from tree limb
  Employees legs crushed by fallen 4000 lb. bundle of steel
       Employee killed when struck by 1,700 lb spreader bar
  Employee sustains head injuries and goes home, later dies
              Employee Fractures Pelvis in Fall from Ladder
                    Employee falls from tree and later dies
One Employee

## 5. Attaching the narrative text

Each accident has a narrative stored as several short lines. I read only the lines for my construction accidents and join them into one text per accident.

In [9]:
cons = base[base["is_construction"]].copy()
cons_ids = set(cons["summary_nr"].dropna().astype("int64"))

parts = []
for chunk in pd.read_sql("SELECT summary_nr, line_nr, abstract_text FROM accident_abstract", conn, chunksize=500_000):
    chunk["summary_nr"] = pd.to_numeric(chunk["summary_nr"], errors="coerce")
    parts.append(chunk[chunk["summary_nr"].isin(cons_ids)])

abstract = pd.concat(parts, ignore_index=True)
abstract["summary_nr"] = abstract["summary_nr"].astype("int64")
abstract["line_nr"] = pd.to_numeric(abstract["line_nr"], errors="coerce")

narratives = (
    abstract.sort_values(["summary_nr", "line_nr"])
    .groupby("summary_nr")["abstract_text"]
    .apply(lambda lines: " ".join(str(l).strip() for l in lines if pd.notna(l)))
    .reset_index()
    .rename(columns={"abstract_text": "full_narrative"})
)
narratives["summary_nr"] = narratives["summary_nr"].astype("Int64")

cons = cons.merge(narratives, on="summary_nr", how="left")
print("Construction rows:", len(cons))
print("Rows with a narrative:", int(cons["full_narrative"].notna().sum()))

Construction rows: 74523
Rows with a narrative: 63449


## 6. Leakage audit on the larger data

I repeat the check from my first pass. For each candidate column I ask how well I could tell fatal from non fatal cases using only whether the column is filled in. A column that does much better than guessing the majority class is revealing the answer, so I leave it out. I also scrub outcome words such as killed and died from the text.

In [10]:
LEAK_WORDS = r"\b(?:died|dead|death|deaths|killed|kill|fatal|fatally|fatality|fatalities|deceased|expired|pronounced)\b"

NUMERIC_COLS = ["age", "nature_of_inj", "part_of_body", "src_of_injury", "event_type", "evn_factor",
                "hum_factor", "occ_code", "task_assigned", "const_op", "const_op_cause", "fat_cause",
                "build_stories", "nonbuild_ht"]
for c in NUMERIC_COLS:
    cons[c] = pd.to_numeric(cons[c], errors="coerce")

CANDIDATES = NUMERIC_COLS + ["const_end_use", "project_cost", "project_type", "hazsub", "fatality"]
cons["is_fatal"] = (cons["degree_of_inj"] == 1).astype(int)
majority = max(cons["is_fatal"].mean(), 1 - cons["is_fatal"].mean())

rows = []
for c in CANDIDATES:
    s = cons[c]
    present = s.notna() & (s.astype(str).str.strip() != "")
    a = (present.astype(int) == cons["is_fatal"]).mean()
    rows.append((c, round(present.mean(), 3), round(max(a, 1 - a), 3)))

audit = pd.DataFrame(rows, columns=["column", "share_filled_in", "acc_from_filled_in_only"])
audit = audit.sort_values("acc_from_filled_in_only", ascending=False)
print("Majority class baseline:", round(majority, 3))
print(audit.to_string(index=False))

has_leak_word = (cons["event_desc"].fillna("") + " " + cons["event_keyword"].fillna("") + " " +
                 cons["full_narrative"].fillna("")).str.lower().str.contains(LEAK_WORDS, regex=True)
print()
print("Share of texts with outcome words, by severity:")
print(has_leak_word.groupby(cons["degree_of_inj"].map(label_map)).mean().round(3))

Majority class baseline: 0.519
        column  share_filled_in  acc_from_filled_in_only
      fatality            0.425                    0.873
 build_stories            0.347                    0.559
  project_type            0.512                    0.548
 const_end_use            0.505                    0.547
 task_assigned            0.940                    0.541
   nonbuild_ht            0.247                    0.530
  project_cost            0.376                    0.525
const_op_cause            0.928                    0.520
      occ_code            0.975                    0.519
           age            1.000                    0.519
    event_type            1.000                    0.518
     fat_cause            0.786                    0.503
      const_op            0.787                    0.503
  part_of_body            0.789                    0.502
 nature_of_inj            0.789                    0.502
    evn_factor            0.789                    0.502


## 7. Building and saving the modeling table

Any column that predicts the outcome from its presence alone by a wide margin (more than 10 points above the majority baseline) is dropped automatically, and I always drop `fatality`. For the remaining coded fields I fill empty values with a marker and add a missingness flag. I scrub outcome words from the text, and I keep the rule columns for reference only, not as features.

In [12]:
AUTO_DROP_MARGIN = 0.10
leaky = audit[audit["acc_from_filled_in_only"] > majority + AUTO_DROP_MARGIN]["column"].tolist()
if "fatality" not in leaky:
    leaky.append("fatality")
print("Columns dropped for leakage:", leaky)

def scrub(text):
    return re.sub(LEAK_WORDS, " ", str(text).lower())

final = cons.copy()
final["event_date"] = pd.to_datetime(final["event_date"], errors="coerce")
final["text_clean"] = (final["event_desc"].fillna("") + " " + final["event_keyword"].fillna("") + " " +
                       final["full_narrative"].fillna("")).map(scrub)
final["severity_3class"] = final["degree_of_inj"].astype(int)

for c in ["const_op_cause", "fat_cause", "const_op", "occ_code"]:
    if c in final.columns and c not in leaky:
        final[f"{c}_missing"] = final[c].isna().astype(int)
        final[c] = final[c].fillna(-1)

final = final.rename(columns={"naics_code": "audit_naics_code", "site_state": "audit_state", "open_date": "audit_open_date"})
for r in ["rule_naics", "rule_sic", "rule_cited_1926", "rule_keyword"]:
    final[f"audit_{r}"] = final[r].astype(int)

drop_cols = [c for c in leaky + [
    "full_narrative", "event_desc", "event_keyword", "hazsub", "sex", "fall_distance", "fall_ht",
    "rel_insp_nr", "injury_line_nr", "load_dt", "activity_nr", "sic_code", "safety_const", "insp_type",
    "rule_naics", "rule_sic", "rule_cited_1926", "rule_keyword", "is_construction",
] if c in final.columns]
final = final.drop(columns=drop_cols).sort_values("event_date").reset_index(drop=True)

out_path = PROC / "accidents_construction_expanded.csv"
final.to_csv(out_path, index=False)

print("Saved:", out_path)
print("Rows:", len(final), "| unique accidents:", final["summary_nr"].nunique())
print("Fatal versus non fatal:", final["is_fatal"].value_counts().to_dict())
print("Three class counts:", final["severity_3class"].map(label_map).value_counts().to_dict())
print("Date range:", final["event_date"].min(), "to", final["event_date"].max())
print()
print("Columns:", final.columns.tolist())

Columns dropped for leakage: ['fatality']
Saved: c:\Users\sabin\constructiq\data\processed\accidents_construction_expanded.csv
Rows: 74523 | unique accidents: 63180
Fatal versus non fatal: {0: 38641, 1: 35882}
Three class counts: {'Fatality': 35882, 'Hospitalized': 31437, 'Not hospitalized': 7204}
Date range: 1976-01-09 00:00:00+00:00 to 2025-03-28 00:00:00+00:00

Columns: ['summary_nr', 'age', 'nature_of_inj', 'part_of_body', 'src_of_injury', 'event_type', 'evn_factor', 'hum_factor', 'occ_code', 'degree_of_inj', 'task_assigned', 'const_op', 'const_op_cause', 'fat_cause', 'audit_naics_code', 'audit_state', 'audit_open_date', 'event_date', 'const_end_use', 'build_stories', 'nonbuild_ht', 'project_cost', 'project_type', 'is_fatal', 'text_clean', 'severity_3class', 'const_op_cause_missing', 'fat_cause_missing', 'const_op_missing', 'occ_code_missing', 'audit_rule_naics', 'audit_rule_sic', 'audit_rule_cited_1926', 'audit_rule_keyword']


In [13]:
# Can I now split by time? Check classes in an 80 / 20 chronological split.
dated = final.dropna(subset=["event_date"]).sort_values("event_date").reset_index(drop=True)
cut = int(len(dated) * 0.8)
print("Rows by year:")
print(dated["event_date"].dt.year.value_counts().sort_index().to_string())
print()
print("Train classes:", dated.iloc[:cut]["severity_3class"].map(label_map).value_counts().to_dict())
print("Test classes:", dated.iloc[cut:]["severity_3class"].map(label_map).value_counts().to_dict())
print("Test period starts:", dated.iloc[cut]["event_date"].date())

Rows by year:
event_date
1976       1
1977       1
1980       3
1981       1
1982       1
1983       8
1984    1066
1985    1469
1986    1378
1987    1400
1988    1433
1989    1529
1990    1641
1991    1703
1992    1525
1993    1415
1994    1611
1995    1450
1996    1447
1997    1555
1998    1587
1999    1768
2000    1830
2001    1959
2002    1951
2003    2045
2004    1981
2005    1996
2006    2007
2007    1954
2008    1838
2009    1336
2010    1236
2011    1166
2012    1220
2013    1051
2014     660
2015     797
2016     724
2017    2642
2018    2772
2019    2790
2020    2187
2021    1055
2022     861
2023     653
2024     652
2025      95

Train classes: {'Fatality': 24056, 'Hospitalized': 21458, 'Not hospitalized': 5246}
Test classes: {'Hospitalized': 6802, 'Fatality': 5032, 'Not hospitalized': 856}
Test period starts: 2017-05-30


In [14]:
conn.close()

## Where this leaves me

I now have a construction accident table built from the full DOL enforcement data instead of a small snapshot. It is selected using industry codes and construction citations rather than keywords alone, and the columns that give away the outcome have been removed and outcome words scrubbed from the text.

The next step is to return to my baselines notebook and train the models on this larger table, using repeated cross validation that keeps all rows from one accident together.